# Suite ANY — Values of any type

`Schemas.OfAny.Value` is the kind of a value of any type: an `OfAny.Builder` with several types selected restricts it
to them, and one with none selected leaves it unrestricted. A restricted value is written bare, its type the one whose
plain form it takes; an unrestricted one is tagged with its type, a scalar's token name or a registered schema's name,
and is assigned as a value, built standalone through the store where it is not a scalar.

In [ ]:
from mbse.Schemas.Framework import Comparison as C, JSON, Modules, Plain, Proxies, Schemas as S, Validators, YAML
from mbse.Schemas.Framework.Errors import DecodeError, ReadOnlyError
from support import raises, root_of, text

Int, Text, Float, Bytes = (S.OfScalar.Data(t) for t in (int, str, float, bytes))
Phone = S.OfObject.Builder().name("Phone").properties(text("number")).create()
Code = S.OfUnion.Builder().name("Code").branches(lambda b: b.name("n").of(Int), lambda b: b.name("s").of(Text)).create()
Stamp = S.OfIntersection.Builder().name("Stamp").parts(lambda p: p.name("at").of(Int), lambda p: p.name("by").of(Text)).create()
Tags = S.OfContainer.Builder().name("Tags").index(Int).element(Text).create()
Labels = S.OfContainer.Builder().name("Labels").index(Text).element(Text).create()  # a dict by text: a mapping
Cell = S.OfObject.Builder().properties(text("r", int), text("c", int)).create()
Grid = S.OfContainer.Builder().index(Cell).element(Int).create()  # a tensor: nested arrays, or flat with its shape
One = S.OfAny.resolve(lambda t: t.as_scalar(int).as_scalar(str).as_object(Phone))
Free = S.OfAny.resolve(lambda t: t)
Card = S.OfObject.Builder().name("Card").ref(r"card-(\d+)").properties(
    lambda p: p.name("one").of(One), lambda p: p.name("free").of(Free),
    lambda p: p.name("many").of(lambda t: t.as_container(lambda c: c.index(Int).element(Free))),
    lambda p: p.name("tags").of(lambda t: t.as_scalar(bool).as_container(Tags))).create()
store = Proxies.OfStore()
for schema in (Phone, Code, Stamp, Tags, Labels, Card):
    store.register(schema)


def held(prop):
    """What a property holds, read through its visitor: a scalar, or a visitor's schema's name; None where absent."""
    found = []
    prop.as_value(lambda a: a.as_scalar(lambda s: found.extend([] if s.get() is None else [s.get()]) or s)
                  .as_object(lambda v: found.extend([v.as_schema().name] if v.as_schema() else []) or v)
                  .as_union(lambda v: found.extend([v.as_schema().name] if v.as_schema() else []) or v)
                  .as_intersection(lambda v: found.extend([v.as_schema().name] if v.as_schema() else []) or v)
                  .as_container(lambda v: found.extend([v.as_schema().name] if v.as_schema() else []) or v))
    return found[0] if found else None

## ANY-01 · An `OfAny` restricted to the types selected, or to none; its types told apart on the wire

In [ ]:
assert type(One) is S.OfAny.Value and One.types == (Int, Text, Phone) and One.validate() == []
assert Free.types == () and Free.validate() == [] and Card.validate() == []
assert S.OfAny.resolve(lambda t: t.as_scalar(int)) == Int  # one type selected: that type, as ever
assert (S.plain_forms(Float), S.plain_forms(Bytes), S.plain_forms(Labels), S.plain_forms(Tags), S.plain_forms(Grid)) == (
    {"float", "text"}, {"text"}, {"mapping"}, {"list"}, {"list", "mapping"})  # NaN is text, bytes base64, a tensor flat
assert S.plain_forms(One) == {"int", "text", "mapping"} and S.plain_forms(Free) == {"mapping"}  # unrestricted: tagged
assert S.plain_forms(S.OfScalar.Data(S.OfScalar.Token("ccpp", "x"))) == {"bool", "int", "float", "text", "mapping", "list"}
loop = S.OfApply.Data()
loop.of = loop
assert S.plain_forms(loop) == frozenset()  # an application that applies itself has no values
assert S.OfAny.Value((loop, S.OfScalar.Data("weird"))).validate() == [
    "a schema: an application cannot apply itself", "a schema: unsupported scalar type 'weird'"]
assert S.OfAny.Value((Tags, Grid)).validate() == ["an OfAny's types Tags and a container are not told apart on the wire"]
assert S.OfAny.Value((Cell, Labels)).validate() == [
    "an OfAny's types a value object and Labels are not told apart on the wire"]
assert One == One and One != Free  # schemas by identity
Either = S.OfUnion.Builder().branches(lambda b: b.name("a").of(Int), lambda b: b.name("b").of(Text)).create()
Both = S.OfIntersection.Builder().parts(lambda p: p.name("a").of(Int), lambda p: p.name("b").of(Text)).create()
assert S.OfAny.Value((Float, Text, Phone, Code, Labels, Either, Both, Tags)).validate() == [
    "an OfAny's types float and str are not told apart on the wire",
    "an OfAny's types Phone and Code are not told apart on the wire",
    "an OfAny's types Phone and Labels are not told apart on the wire",
    "an OfAny's types Phone and a union are not told apart on the wire",
    "an OfAny's types Phone and an intersection are not told apart on the wire",
    "an OfAny's types Code and Labels are not told apart on the wire",
    "an OfAny's types Code and a union are not told apart on the wire",
    "an OfAny's types Code and an intersection are not told apart on the wire",
    "an OfAny's types Labels and a union are not told apart on the wire",
    "an OfAny's types Labels and an intersection are not told apart on the wire",
    "an OfAny's types a union and an intersection are not told apart on the wire"]
assert S.OfAny.Value((3, One, S.OfScalar.Data(int, name="1x"), Grid)).validate() == [
    "an OfAny's types are schemas, got 3", "an OfAny's types are schemas of other kinds, not an OfAny",
    "1x: name '1x' is not identifiers separated by dots, e.g. 'crm.Contact'"]
Thing = S.OfObject.Builder().name("Thing").ref().create()
assert S.OfObject.Builder().properties(lambda p: p.name("x").of(lambda t: t.as_scalar(int).as_object(Thing))).create(
    ).validate() == ["property 'x': a reference object schema cannot be a property's type"]  # a value only

## ANY-02 · Proxies: a restricted value written through the visitors as its type is, inferred; an unrestricted one given as a value, built standalone where it is not a scalar

In [ ]:
card = store.Card().one(3).free("x").create()
assert (held(card.one), held(card.free)) == (3, "x") and card.one.get() == 3
card = store.Card().one(lambda p: p.as_object(lambda o: o.number("1"))).tags(  # the one type of its kind
    lambda p: p.as_value(lambda a: a.as_container(lambda t: t.at(0).as_scalar("a")))).create()
assert (held(card.one), held(card.tags)) == ("Phone", "Tags") and card.one.get() is None
with raises(TypeError, match="property 'one' does not hold a value of Tags"):
    store.Card().one(store.Tags().create())
with raises(TypeError, match="'Phone' is a value object schema; build a value through its owner, or standalone through "
                            "the store"):
    Proxies.OfObject.Builder(store, Phone, "Phone")
with raises(ReadOnlyError, match="property 'one' does not hold a union value"):
    store.Card().one(lambda p: p.as_value(lambda a: a.as_union(lambda u: u.n(1))))
# Unrestricted: a scalar, or a value, standalone or another's; a Spec has no schema to build by.
phone = store.Phone().number("2").create()
tags = store.Tags()
tags.at(0).as_scalar("t")
tags = tags.create()
assert Proxies.owner_of(phone) is None and tags.as_schema() is Tags
values = [phone, store.Code().n(1).create(), store.Stamp().at(1).by("me").create(), tags, card.one]
assert [held(store.Card().free(value).create().free) for value in values] == ["Phone", "Code", "Stamp", "Tags", "Phone"]
assert held(store.Card().free(lambda p: p.as_value(lambda a: a.as_object(card.one))).create().free) == "Phone"
fresh = store.Phone().number("3").create()
first, second = store.Card().free(fresh).create(), store.Card().free(fresh).create()
phones = []
for each in (first, second):
    each.free.as_value(lambda a: a.as_object(lambda o: phones.append(o) or o))
assert phones[0] is fresh and phones[1] is not fresh and Proxies.owner_of(fresh) is first  # adopted once, then copied
unbuilt = "property 'free' holds a value of any type: give a value, which carries its schema, not a Spec"
with raises(ReadOnlyError, match=unbuilt):
    store.Card().free(lambda p: p.as_value(lambda a: a.as_object(lambda o: o.number("1"))))
with raises(ReadOnlyError, match=unbuilt):
    store.Card().free(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.remove(0))))
read = []  # a Spec of another kind than the value held reads it as empty, and writes nothing
store.Card().one(3).one(lambda p: p.as_value(lambda a: a.as_object(lambda o: read.append(o.as_schema()) or o))).create()
assert read == [None]
found = []
card.free.as_value(lambda a: a.as_scalar(lambda s: found.append((s.as_schema(), s.get())) or s))
assert found == [(Free, None)]  # nothing held: a scalar of any type, absent
listed = store.Card().many(lambda p: p.as_value(lambda a: a.as_container(
    lambda c: c.at(0).as_scalar(1) and c.at(1).as_object(phone)))).create()
elements = []
listed.many.as_value(lambda a: a.as_container(lambda c: c.as_entries(elements.extend)))
assert [index for index, _ in elements] == [0, 1]
with raises(TypeError, match="property 'one' does not hold a value of a container"):
    store.Card().one(listed.many)

## ANY-03 · Snapshots: a restricted value bare, an unrestricted one tagged, read back; malformed ones refused, with a path

In [ ]:
other = Proxies.OfStore()
for schema in (Phone, Code, Stamp, Tags, Labels, Card):
    other.register(schema)


def many(c):
    c.at(0).as_scalar(2.5)
    c.at(1).as_object(phone)
    c.at(3).as_container(tags)  # nothing at 2
    return c


card = (store.Card().one(lambda p: p.as_object(lambda o: o.number("1"))).free(store.Code().s("c").create())
        .many(lambda p: p.as_value(lambda a: a.as_container(many))).tags(True).create())
written = root_of(Plain.ToPlain(store).OfObject(Card, card))
assert written == {"one": {"number": "1"}, "free": {"Code": {"s": "c"}}, "tags": True,
                   "many": [{"float": 2.5}, {"Phone": {"number": "2"}}, None, {"Tags": ["t"]}]}
for encode, decode in [(JSON.ToJSON, JSON.FromJSON), (YAML.ToYAML, YAML.FromYAML)]:
    text_ = encode(store).OfObject(Card, card)
    assert encode(other).OfObject(Card, decode(other).OfObject(Card, text_)) == text_
assert root_of(Plain.ToPlain(store).OfObject(Card, store.Card().one("s").free(b"\x01").create())) == {
    "one": "s", "free": {"bytes": "AQ=="}}
assert Plain.FromPlain(store).OfObject(Card, Plain.ToPlain(store).OfObject(Card, card)) is card  # reloading: an update
back = Plain.FromPlain(other).OfObject(Card, {"root": "Card/card-8", "objects": {"Card/card-8": {"one": 3, "tags": ["a"]}}})
assert (held(back.one), held(back.tags)) == (3, "Tags")  # each read as the one type its plain form fits
assert held(Plain.FromPlain(other).OfObject(Card, {"root": "Card/card-7", "objects": {"Card/card-7": {"one": "s"}}}).one) == "s"


def refused(values, message):
    with raises(DecodeError, match='$.objects["Card/card-9"]' + message):
        Plain.FromPlain(other).OfObject(Card, {"root": "Card/card-9", "objects": {"Card/card-9": values}})


refused({"one": True}, ".one: a value of any type takes the plain form of one of its types, got bool")
refused({"one": 2.5}, ".one: a value of any type takes the plain form of one of its types, got float")
refused({"one": None}, ".one: a value of any type takes the plain form of one of its types, got NoneType")
refused({"free": 3}, ".free: a value of any type is {type: value}, its type a scalar's token name or a registered "
                     "schema's name")
refused({"free": {"Nope": 1}}, ".free.Nope: no schema registered as 'Nope'")
refused({"free": {"Card": {}}}, ".free.Card: 'Card' is not a value's schema")
refused({"free": {"int": "x"}}, ".free.int: expected int, got str")
with raises(TypeError, match="a value of any type holds one of its types; float is not one"):
    Plain.ToPlain(store).OfObject(Card, store.Card().one(2.5).create())  # builders do not check scalars' types
Point = S.OfObject.Builder().name("Point").properties(text("x", int)).create()  # not registered
Spot = S.OfObject.Builder().name("Spot").ref().properties(lambda p: p.name("at").of(Point)).create()
store.register(Spot)
loose = store.Card().free(store.Spot().at(lambda p: p.as_object(lambda o: o.x(1))).create().at).create()
with raises(TypeError, match="a value of any type is of a registered schema; 'Point' is not registered"):
    Plain.ToPlain(store).OfObject(Card, loose)

## ANY-04 · Validation and comparison under the type a value holds; uniqueness by value

In [ ]:
validate = Validators.Validate(store)
assert validate(Card, card) == []
assert validate(Card, store.Card().one(2.5).many(lambda p: p.as_value(lambda a: a.as_container(
    lambda c: c.at(0).as_object(phone)))).create()) == ["Card#0.one: float is not one of its types"]
assert validate(Card, loose) == ["Card#0.free: a value of any type is of a registered schema; 'Point' is not registered"]
assert validate(Card, store.Card().free(store.Code().n("x").create()).create()) == ["Card#0.free.n: expected int, got str"]


def compare(a, b):
    return C.OfObject(Card, a).compare(C.OfObject(Card, b))


assert compare(store.Card().one(3).free(phone).create(), store.Card().one(3).free(store.Phone().number("2").create()).create()) == 0
assert compare(store.Card().one(3).create(), store.Card().one("3").create()) is None  # of different types: incomparable
assert compare(store.Card().free(phone).create(), store.Card().free(store.Code().s("2").create()).create()) is None
Holds = S.OfRelation.Builder().name("Holds").links("card", "other").properties(
    lambda p: p.name("what").of(Free), text("note")).unique("note").create()
Holder = S.OfObject.Builder().name("Holder").ref().relations(lambda r: r.name("holds").of(Holds).me("card"),
                                                              lambda r: r.name("held").of(Holds).me("other")).create()
for schema in (Holds, Holder):
    store.register(schema)
target = store.Holder().create()
holder = store.Holder().holds(lambda a: a.insert(lambda e: e.other(target).what(phone).note("a")).insert(
    lambda e: e.other(target).what(store.Phone().number("2").create()).note("b")).insert(
    lambda e: e.other(target).what(1).note("c")).insert(lambda e: e.other(target).note("d"))).create()
assert validate(Holder, holder) == [
    "unique(note) violated: entries agreeing on ['card', 'other', 'what'] differ on ['note']"]  # equal values: one key
for schema in (Holds, Holder):
    other.register(schema)
snapshot = Plain.ToPlain(store).Reachable(Holder, holder)
assert [e.get("what") for e in root_of(snapshot)["holds"]] == [{"Phone": {"number": "2"}}] * 2 + [{"int": 1}, None]
assert Plain.ToPlain(other).Reachable(Holder, Plain.FromPlain(other).Reachable(Holder, snapshot)) == snapshot
assert Plain.FromPlain(store).Reachable(Holder, snapshot) is holder and len(holder.holds) == 4  # reloading adds none
assert C.OfObject(Card, card).compare(C.OfObject(Card, Plain.FromPlain(other).OfObject(Card, Plain.ToPlain(store).OfObject(
    Card, card)))) == 0  # a container of values of any type, element by element

## ANY-05 · Modules write a value of any type's types, by name or inline, and read them back

In [ ]:
module = Modules.module(store, [Phone, Card])
definition = {e: d for e, d in root_of(Plain.ToPlain(store)(S.Module.Schema, module))["schemas"].items()}["Card"]
assert definition["object"]["properties"][:2] == [
    {"name": "one", "type": {"any": {"types": [{"scalar": {"format": "basic", "token": "int"}},
                                               {"scalar": {"format": "basic", "token": "str"}},
                                               {"named": {"name": "Phone"}}]}}},
    {"name": "free", "type": {"any": {}}}]
back = Modules.schemas(store, module)
one, free = (back["Card"].properties[n].type for n in ("one", "free"))
assert type(one) is S.OfAny.Value and one.types[2] is back["Phone"] and one.types[0] == Int and free.types == ()
assert Modules.reference(S.OfAny.Value((Int,), description="Counted")) == {"any": {"types": [
    {"scalar": {"format": "basic", "token": "int"}}], "description": "Counted"}}